### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
sys.path.append('/home/vino/.cache/huggingface/hub')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from siglip_class import SVGMetricEvaluator

### Random seed for reproducibility

In [3]:
import torch
import random
import numpy as np
import multiprocessing as mp

mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc


class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i2000_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-21 20:37:28 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
model=Model()

WARNING 04-21 20:37:28 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-21 20:37:33 [config.py:585] This model supports multiple tasks: {'classify', 'reward', 'embed', 'score', 'generate'}. Defaulting to 'generate'.
INFO 04-21 20:37:33 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-21 20:37:34 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i2000_msl2048', speculative_config=None, tokenizer='./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i2000_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=Decodi

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-21 20:37:46 [loader.py:447] Loading weights took 11.09 seconds
INFO 04-21 20:37:46 [gpu_model_runner.py:1186] Model loading took 14.2487 GB and 11.240056 seconds
INFO 04-21 20:37:53 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/d8d91d95bf/rank_0_0 for vLLM's torch.compile
INFO 04-21 20:37:53 [backends.py:425] Dynamo bytecode transform time: 6.74 s
INFO 04-21 20:37:55 [backends.py:132] Cache the graph of shape None for later use


[rank0]:W0421 20:37:56.497000 19386 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 04-21 20:38:12 [backends.py:144] Compiling a graph for general shape takes 18.66 s
ERROR 04-21 20:38:14 [core.py:343] EngineCore hit an exception: Traceback (most recent call last):
ERROR 04-21 20:38:14 [core.py:343]   File "/home/vino/anaconda3/envs/kaggle/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 335, in run_engine_core
ERROR 04-21 20:38:14 [core.py:343]     engine_core = EngineCoreProc(*args, **kwargs)
ERROR 04-21 20:38:14 [core.py:343]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ERROR 04-21 20:38:14 [core.py:343]   File "/home/vino/anaconda3/envs/kaggle/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 290, in __init__
ERROR 04-21 20:38:14 [core.py:343]     super().__init__(vllm_config, executor_class, log_stats)
ERROR 04-21 20:38:14 [core.py:343]   File "/home/vino/anaconda3/envs/kaggle/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 63, in __init__
ERROR 04-21 20:38:14 [core.py:343]     num_gpu_blocks, num_cpu_blocks = self._initial

In [ ]:
#tmp=model.predict(['sun rising in the east','A golden goose with a fish'])

In [ ]:
#print(tmp[0])

In [ ]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

In [ ]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [ ]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 12
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


In [ ]:
df['svg_3']=results

In [ ]:
model.close_model()

In [ ]:
#SigLip Score
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

In [ ]:
df['svg_score_3'].mean()